In [ ]:
!pip install gradio python-docx PyMuPDF pandas google-generativeai

# Imports
import gradio as gr
import docx
import fitz  # PyMuPDF
import pandas as pd
import re
import os
import google.generativeai as genai
import time

# --- Text Extraction Helpers ---
def extract_text_from_docx(path):
    doc = docx.Document(path)
    return ' '.join([para.text.strip() for para in doc.paragraphs if para.text.strip()])

def extract_text_from_pdf(filepath):
    with fitz.open(filepath) as doc:
        return ' '.join([page.get_text() for page in doc])

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# --- Generate Short Summary for Risk Detection ---
def generate_short_summary(text):
    if len(text) > 800:
        text = text[:800]
    return text

# --- Smart Field Extraction ---
def extract_fields(text):
    fields = {}

    # Court
    court_match = re.search(r'(Local Court|Regional Court|Higher Regional Court|Landgericht|Amtsgericht)\s+of\s+([\w\s]+)', text)
    fields['Court'] = court_match.group(0) if court_match else "Not Found"

    # File Number
    file_match = re.search(r'File Name:\s*(\w+)', text)
    fields['File Number'] = file_match.group(1) if file_match else "Not Found"

    # Claimant Name & Address
    claimant_match = re.search(r'Lawsuit\s*of\s*(Mr\.|Ms\.|Mrs\.)\s*([^,]+),\s*([^\n]+)', text)
    if claimant_match:
        fields['Name of Claimant'] = claimant_match.group(2).strip()
        fields['Address of Claimant'] = claimant_match.group(3).strip()
    else:
        fields['Name of Claimant'] = "Not Found"
        fields['Address of Claimant'] = "Not Found"

    fields['Number of Claimants'] = "1"
    fields['Type of Claimant (Natural vs Legal Person)'] = "Natural Person"

    # Representation
    if 'No legal representative' in text:
        fields['Is the Claimant Represented?'] = "No"
        fields['Name of the Legal Representative'] = "None"
        fields['Address of the Legal Representative'] = "None"
    else:
        rep_match = re.search(r'Legal Representative:\s*(?:Attorney\s*)?(Dr\.)?\s*([^,\n]+),\s*([^\n]+)', text)
        if rep_match:
            fields['Is the Claimant Represented?'] = "Yes"
            fields['Name of the Legal Representative'] = rep_match.group(2).strip()
            fields['Address of the Legal Representative'] = rep_match.group(3).strip()
        else:
            fields['Is the Claimant Represented?'] = "No"
            fields['Name of the Legal Representative'] = "None"
            fields['Address of the Legal Representative'] = "None"

    # Money
    money_match = re.search(r'EUR\s*([0-9,.]+)', text)
    fields['Money'] = f"EUR {money_match.group(1)}" if money_match else "Not Found"

    # Action
    lower_text = text.lower()
    if 'subsequent performance' in lower_text:
        fields['Action'] = "Subsequent Performance"
    elif 'withdrawal' in lower_text:
        fields['Action'] = "Withdrawal Confirmation"
    elif 'pain and suffering' in lower_text or 'immaterial damages' in lower_text:
        fields['Action'] = "Compensation for Injury"
    else:
        fields['Action'] = "Other"

    fields['Remove a Disturbance'] = "Yes" if any(word in lower_text for word in ['defect', 'repair', 'remove disturbance']) else "No"
    fields['Desist'] = "No"

    # BMW involvement
    fields['BMW Involved?'] = "Yes" if 'BMW' in text else "No"
    fields['BMW w Non-BMW'] = "BMW"
    fields['BMW and Other Parties'] = "Only BMW"

    # Legal Basis
    basis_match = re.findall(r'\u00a7\s*[0-9]+(?:\s*(?:para\.|Abs\.)\s*[0-9]+)?\s*(?:BGB|ProdHaftG)', text)
    fields['Legal Basis for the Claim'] = ', '.join(basis_match) if basis_match else "Not Found"

    if 'purchase contract' in lower_text or 'leasing agreement' in lower_text:
        fields['Contractual Claims'] = "Yes"
    else:
        fields['Contractual Claims'] = "No"

    if 'BGB' in text or 'ProdHaftG' in text:
        fields['Statutory Claims'] = "Yes"
    else:
        fields['Statutory Claims'] = "No"

    fields['Procedural Claims'] = "Yes" if 'costs of the proceedings' in lower_text else "No"

    return fields

# --- Risk Detection using Gemini (Safe Fallback Version) ---
def detect_risks_gemini_safe(summary, api_key):
    genai.configure(api_key=api_key)
    model = genai.GenerativeModel('gemini-1.5-flash')

    prompt = f"""
Analyze this short summary and determine if each of the following risks are present:
- Normative Risks
- Evidentiary Risks
- Procedural Risks
- Relational Risks
- Systemic Risks
- Material Risks

For each category, reply exactly like:
Normative Risks: ✅ or ⬜
Evidentiary Risks: ✅ or ⬜
Procedural Risks: ✅ or ⬜
Relational Risks: ✅ or ⬜
Systemic Risks: ✅ or ⬜
Material Risks: ✅ or ⬜

Summary:
{summary}
"""

    try:
        start_time = time.time()

        response = model.generate_content(prompt)
        elapsed = time.time() - start_time

        if elapsed > 20:  # Timeout of 20 seconds
            raise TimeoutError("Risk detection skipped due to server delay.")

        output = response.text
        risks = {}
        for line in output.strip().split('\n'):
            if ':' in line:
                category, mark = line.split(':', 1)
                risks[category.strip()] = mark.strip()
        return risks

    except Exception as e:
        # If timeout or any error → fallback
        risks = {
            "Normative Risks": "Server Slow ❌",
            "Evidentiary Risks": "Server Slow ❌",
            "Procedural Risks": "Server Slow ❌",
            "Relational Risks": "Server Slow ❌",
            "Systemic Risks": "Server Slow ❌",
            "Material Risks": "Server Slow ❌"
        }
        return risks

# --- Upload Handler ---
def upload_and_extract(file, api_key):
    if file is None:
        return pd.DataFrame([{"Field": "Error", "Extracted Information": "No file uploaded."}]), pd.DataFrame()

    filepath = file.name if hasattr(file, 'name') else file

    try:
        if filepath.endswith('.txt'):
            with open(filepath, 'r', encoding='utf-8') as f:
                document_text = f.read()
        elif filepath.endswith('.docx'):
            document_text = extract_text_from_docx(filepath)
        elif filepath.endswith('.pdf'):
            document_text = extract_text_from_pdf(filepath)
        else:
            return pd.DataFrame([{"Field": "Error", "Extracted Information": "Unsupported file format."}]), pd.DataFrame()

        document_text = clean_text(document_text)
        fields = extract_fields(document_text)

        # 📋 Generate small summary
        short_summary = generate_short_summary(document_text)

        # 🔥 Risk Detection with fallback
        risks = detect_risks_gemini_safe(short_summary, api_key)

        output_fields_df = pd.DataFrame(list(fields.items()), columns=["Field", "Extracted Information"])
        output_risks_df = pd.DataFrame(list(risks.items()), columns=["Risk Category", "Detected"])

        return output_fields_df, output_risks_df

    except Exception as e:
        return pd.DataFrame([{"Field": "Error", "Extracted Information": str(e)}]), pd.DataFrame()

# --- Gradio Interface ---
with gr.Blocks() as demo:
    gr.Markdown("# 🧾 Legal Claim Extractor - Gemini Fallback Edition 🚀")

    api_key = gr.Textbox(label="Enter your Google Gemini API Key", type="password")
    uploader = gr.File(label="Upload Legal Document", file_types=[".pdf", ".docx", ".txt"])

    output_table = gr.DataFrame(label="Extracted Fields")
    risk_table = gr.DataFrame(label="Detected Risk Categories (or Server Status)")

    upload_btn = gr.Button("Analyze")
    upload_btn.click(upload_and_extract, inputs=[uploader, api_key], outputs=[output_table, risk_table])

    demo.launch(share=True)


































































Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://afc0211317c15e660c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
